In [77]:
import numpy as np
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.model_selection import LeaveOneOut
import pandas as pd


import os

In [16]:
df_clip_base = pd.read_csv("confdf_results_csv/clip_base-clip.csv", index_col=0)
df_clip_large = pd.read_csv("confdf_results_csv/clip_large-clip_large.csv", index_col=0)

clip_results = {}
clip_large_results = {}

clip_threshold = 0.995
clip_large_threshold = 0.991

for video in df_clip_base["video_name"]:
    if df_clip_base[df_clip_base["video_name"] == video]["raw_pred_mean"].squeeze() < clip_threshold:
        clip_results[video] = 0
    else:
        clip_results[video] = 1

for video in df_clip_large["video_name"]:
    if df_clip_large[df_clip_large["video_name"] == video]["raw_pred_mean"].squeeze() < clip_large_threshold:
        clip_large_results[video] = 0
    else:
        clip_large_results[video] = 1

for video in clip_results.keys():
    if clip_results[video] != clip_large_results[video]:
        print(f"mismatch for {video} - base: {df_clip_base[df_clip_base["video_name"] == video]["raw_pred_mean"].squeeze()} - large: {df_clip_large[df_clip_large["video_name"] == video]["raw_pred_mean"].squeeze()}")

mismatch for 1eec2811670a1dfa0f697d099b0dfcdb_1920x1080_30 - base: 0.9712911 - large: 0.99879193
mismatch for 6cb2bf5384212402d39fc2653df913b7_1920x1080_30 - base: 0.9614033 - large: 0.9966918
mismatch for bruce_willis_1 - base: 0.9389125 - large: 0.9995159
mismatch for e9d9cee75745b4cc77e149448d8855d3_1080x1920_30 - base: 0.999229 - large: 0.95042706
mismatch for e47c246f3c2f0f66eca1a08aea644c57_1080x1920_30 - base: 0.9996847 - large: 0.97866666
mismatch for lucy_liu_3 - base: 0.98080003 - large: 0.9915848


In [49]:

df_results = pd.read_csv(f"confdf_results_csv/xception-xception_best.csv", index_col=0)
df_labels = df_results[["video_name", "label"]]


df_labels.to_csv("confdf_info_csv/videos_labels.csv", index=False)

In [ ]:
selected_models = [
    'recce-recce_best',
    #'srm-srm_best',
    'xception-xception_best',
    #'clip_large-clip_large',
    'clip_base-clip'
    'spsl-spsl_best',
    'ucf-ucf_best',
    'effort-effort_ff_best'
    ]

df_video_by_detectors = pd.read_csv("celebdf_info_csv/videos_labels.csv")


for file in os.listdir("confdf_results_csv"):
    if file.replace(".csv", "") in selected_models:

        df_results = pd.read_csv(f"confdf_results_csv/{file}", index_col=0)

        df_video_by_detectors[f"{file.split("-")[0]}_pred_mean"] = df_results["raw_pred_mean"]

#df_video_by_detectors = df_video_by_detectors.sort_values(["label", "video_name"])

display(df_video_by_detectors)

df_video_by_detectors.to_csv("celebdf_info_csv/videos_by_detectors.csv", index=False)

,video_name,label,recce_pred_mean,srm_pred_mean,effort_pred_mean,xception_pred_mean,ucf_pred_mean,clip_large_pred_mean,spsl_pred_mean
0,nile_red_2,1,0.999882,0.789314,0.980044,0.999286,0.999191,0.999683,0.999323
1,dd5599c094537cb9fd32f33f43c06f87_1920x1080_30,0,0.013265,0.339840,0.377038,0.029065,0.008577,0.018170,0.104179
2,talyor_swift_3,1,0.998343,0.665849,0.977920,0.994439,0.998743,0.999995,0.985816
3,70e80388aae75b945e5000eb7ade4a6a_1920x1080_30,0,0.647860,0.579011,0.455806,0.690315,0.347482,0.122754,0.870809
4,will_smith_2,1,0.999314,0.708435,0.975835,0.999211,0.999177,0.999990,0.996915
...,...,...,...,...,...,...,...,...,...
79,bruce_willis_2,1,0.949844,0.672146,0.963522,0.944011,0.994007,0.999964,0.996553
80,a4ea08e72606fb7ebd4347d79262823f_1920x1080_30,0,0.036198,0.355970,0.803426,0.063876,0.086554,0.352502,0.129158
81,d02c5cdaee1215360fb80240d7a8901f_1080x1920_30,0,0.188374,0.422423,0.389280,0.212857,0.180960,0.737553,0.425727
82,9bd3195d97edad5f200b325825dbb729_1920x1080_30,0,0.001499,0.258457,0.361385,0.024312,0.003133,0.363918,0.230942


In [ ]:

# X : Tableau (84, 6) -> Moyenne des prédictions des 6 détecteurs par vidéo
# y : Vecteur (84,) -> Vérité terrain (0 ou 1)
#X = df_video_by_detectors.drop(columns=["video_name", "label", "srm_pred_mean", "xception_pred_mean", "ucf_pred_mean", "clip_large_pred_mean", "spsl_pred_mean"])
X = df_video_by_detectors.drop(columns=["video_name", "label"])
y = df_video_by_detectors["label"]

# Utilisation de LogisticRegressionCV :
# 1. cv=LeaveOneOut() est parfait pour tes 84 points.
# 2. penalty='l2' empêche les poids de devenir trop extrêmes.
# 3. Cs=10 teste 10 forces de régularisation différentes pour trouver la meilleure.

model = LogisticRegressionCV(
    cv=LeaveOneOut(),
    penalty='l2',
    Cs=10, 
    random_state=49
)

model.fit(X, y)

# --- Analyse des résultats ---
print(f"Précision moyenne (LOOCV) : {model.scores_[1].mean():.2%}")
print("\nPoids des détecteurs :")
for i, weight in enumerate(model.coef_[0]):
    print(f"Détecteur {i+1} : {weight:.4f}")

print(f"Constante (Biais) : {model.intercept_[0]:.4f}")

print("LASSO")

model_lasso = LogisticRegression(penalty='l1', solver='liblinear', C=100)
model_lasso.fit(X, y)

for i, weight in enumerate(model_lasso.coef_[0]):
    print(f"Détecteur {i+1} : {weight:.4f}")
print(f"Constante (Biais) : {model_lasso.intercept_[0]:.4f}")

# On valide avec un Stratified 5-Fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1342)
scores = cross_val_score(model_lasso, X, y, cv=cv)

print(f"Précision moyenne (5-Fold) : {scores.mean():.2%}")

Précision moyenne (LOOCV) : 76.79%

Poids des détecteurs :
Détecteur 1 : 1.2769
Détecteur 2 : 0.5969
Détecteur 3 : 0.9815
Détecteur 4 : 1.2134
Détecteur 5 : 0.9589
Détecteur 6 : 0.6140
Constante (Biais) : -3.7450
LASSO
Détecteur 1 : 11.6924
Détecteur 2 : 1.1924
Détecteur 3 : 0.8438
Détecteur 4 : 0.2729
Détecteur 5 : 4.7509
Détecteur 6 : -3.6230
Constante (Biais) : -10.0124
Précision moyenne (5-Fold) : 92.94%


In [99]:
# 1. Obtenir les prédictions et les probabilités
y_pred = model_lasso.predict(X)
y_proba = model_lasso.predict_proba(X)[:, 1]

# 2. Identifier les erreurs
# Si y est une série Pandas, on utilise .values pour éviter les problèmes d'index
errors_idx = np.where(y_pred != np.array(y))[0]

print(f"--- Analyse des {len(errors_idx)} erreurs ---")
print(f"{'Vidéo ID':<10} | {'Vérité':<8} | {'Pred Lasso':<12} | {'Confiance':<10} | {'Score D1':<8}")
print("-" * 75)

for idx in errors_idx:
    verite = np.array(y)[idx]
    pred = y_pred[idx]
    
    # Calcul de la confiance (probabilité de la classe prédite)
    confiance = y_proba[idx] if pred == 1 else 1 - y_proba[idx]
    
    # Correction de l'accès à D1 : .iloc[ligne, colonne] pour Pandas
    # On force la conversion en numpy pour être tranquille
    if hasattr(X, 'iloc'):
        score_d1 = X.iloc[idx, 0]
    else:
        score_d1 = X[idx, 0]
    
    print(f"{idx:<10} | {verite:<8} | {pred:<12} | {confiance:.2%}   | {score_d1:.2%}")

--- Analyse des 3 erreurs ---
Vidéo ID   | Vérité   | Pred Lasso   | Confiance  | Score D1
---------------------------------------------------------------------------
9          | 0        | 1            | 96.76%   | 83.70%
42         | 1        | 0            | 69.45%   | 68.93%
83         | 1        | 0            | 53.63%   | 48.13%
